In [1]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Cervical Cancer"
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "cnn"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

2025-09-24 22:34:13.233769: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-24 22:34:14.301749: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Found 25000 files belonging to 5 classes.
Using 21250 files for training.


2025-09-24 22:34:18.879730: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 961 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:05:00.0, compute capability: 8.6
2025-09-24 22:34:18.881609: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 21964 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:86:00.0, compute capability: 8.6
2025-09-24 22:34:18.883139: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 21964 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:89:00.0, compute capability: 8.6
2025-09-24 22:34:18.884832: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 21964 MB memory:  -> device: 3, name: NVIDIA GeForce RTX 3090, 

Found 25000 files belonging to 5 classes.
Using 3750 files for validation.
Classes: ['cervix_dyk', 'cervix_koc', 'cervix_mep', 'cervix_pab', 'cervix_sfi']
Epoch 1/20


2025-09-24 22:34:22.000078: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inbaseline_cnn/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2025-09-24 22:34:32.271515: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 175 of 1000
2025-09-24 22:34:42.231284: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 415 of 1000
2025-09-24 22:34:52.300774: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 720 of 1000
2025-09-24 22:35:02.194398: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 962 of 1000
2025-09-24 22:35:04.396429: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] Shuffle buffer filled.
2025-09-24 22:35:05.176094:

2657/2657 [==============================] - 140s 35ms/step - loss: 1.6102 - accuracy: 0.2000 - val_loss: 1.5985 - val_accuracy: 0.0000e+00
Epoch 2/20
2657/2657 [==============================] - 55s 20ms/step - loss: 1.6098 - accuracy: 0.1962 - val_loss: 1.6119 - val_accuracy: 0.0000e+00
Epoch 3/20
2657/2657 [==============================] - 54s 19ms/step - loss: 1.6097 - accuracy: 0.1990 - val_loss: 1.5900 - val_accuracy: 1.0000
Epoch 4/20
2657/2657 [==============================] - 55s 20ms/step - loss: 1.6097 - accuracy: 0.1974 - val_loss: 1.5859 - val_accuracy: 0.0000e+00
Epoch 5/20
2657/2657 [==============================] - 55s 20ms/step - loss: 1.6096 - accuracy: 0.1963 - val_loss: 1.5916 - val_accuracy: 0.0000e+00
Epoch 6/20
2657/2657 [==============================] - 55s 20ms/step - loss: 1.6097 - accuracy: 0.1977 - val_loss: 1.5956 - val_accuracy: 1.0000
Epoch 7/20
2657/2657 [==============================] - 55s 20ms/step - loss: 1.6096 - accuracy: 0.2018 - val_loss: 1.

In [1]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Cervical Cancer"
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "resnet"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

2025-09-24 22:47:21.333757: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-24 22:47:22.397298: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Found 25000 files belonging to 5 classes.
Using 21250 files for training.


2025-09-24 22:47:27.755726: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22280 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:05:00.0, compute capability: 8.6
2025-09-24 22:47:27.757429: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22280 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:86:00.0, compute capability: 8.6
2025-09-24 22:47:27.759061: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 22280 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:89:00.0, compute capability: 8.6
2025-09-24 22:47:27.760597: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 22280 MB memory:  -> device: 3, name: NVIDIA GeForce RTX 3090

Found 25000 files belonging to 5 classes.
Using 3750 files for validation.
Classes: ['cervix_dyk', 'cervix_koc', 'cervix_mep', 'cervix_pab', 'cervix_sfi']
Epoch 1/20


2025-09-24 22:47:49.864362: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 241 of 1000
2025-09-24 22:47:59.936421: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 508 of 1000
2025-09-24 22:48:09.854698: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 738 of 1000
2025-09-24 22:48:19.848722: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 991 of 1000
2025-09-24 22:48:20.094638: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] Shuffle buffer filled.
2025-09-24 22:48:21.331713: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600
2025-09-24 22:48:21.736314: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged 

2657/2657 [==============================] - 153s 40ms/step - loss: 1.4433 - accuracy: 0.3742 - val_loss: 1.1516 - val_accuracy: 0.5481
Epoch 2/20
2657/2657 [==============================] - 52s 18ms/step - loss: 1.3100 - accuracy: 0.4540 - val_loss: 0.8669 - val_accuracy: 0.8435
Epoch 3/20
2657/2657 [==============================] - 51s 18ms/step - loss: 1.2691 - accuracy: 0.4785 - val_loss: 0.9644 - val_accuracy: 0.7527
Epoch 4/20
2657/2657 [==============================] - 48s 17ms/step - loss: 1.2436 - accuracy: 0.4939 - val_loss: 0.9397 - val_accuracy: 0.7281
Epoch 5/20
2657/2657 [==============================] - 51s 18ms/step - loss: 1.2202 - accuracy: 0.5072 - val_loss: 0.8429 - val_accuracy: 0.7714
Epoch 6/20
2657/2657 [==============================] - 50s 18ms/step - loss: 1.2050 - accuracy: 0.5135 - val_loss: 0.9536 - val_accuracy: 0.6747
resnet50  |  Test acc: 0.8557  |  Test loss: 0.8636


In [2]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Cervical Cancer"
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "vgg"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 25000 files belonging to 5 classes.
Using 21250 files for training.
Found 25000 files belonging to 5 classes.
Using 3750 files for validation.
Classes: ['cervix_dyk', 'cervix_koc', 'cervix_mep', 'cervix_pab', 'cervix_sfi']
Epoch 1/20
2657/2657 [==============================] - 61s 21ms/step - loss: 0.8654 - accuracy: 0.6937 - val_loss: 0.3732 - val_accuracy: 0.8536
Epoch 2/20
2657/2657 [==============================] - 62s 22ms/step - loss: 0.6212 - accuracy: 0.7846 - val_loss: 0.2435 - val_accuracy: 0.9338
Epoch 3/20
2657/2657 [==============================] - 61s 22ms/step - loss: 0.5643 - accuracy: 0.8038 - val_loss: 0.2256 - val_accuracy: 0.9396
Epoch 4/20
2657/2657 [==============================] - 63s 23ms/step - loss: 0.5249 - accuracy: 0.8132 - val_loss: 0.2362 - val_accuracy: 0.9225
Epoch 5/20
2657/2657 [==============================] - 63s 23ms/step - loss: 0.4972 - accuracy: 0.8234 - val_loss: 0.1395 - val_accuracy: 0.9754
Epoch 6/20
2657/2657 [===================

In [3]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Cervical Cancer"
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "inception"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 25000 files belonging to 5 classes.
Using 21250 files for training.
Found 25000 files belonging to 5 classes.
Using 3750 files for validation.
Classes: ['cervix_dyk', 'cervix_koc', 'cervix_mep', 'cervix_pab', 'cervix_sfi']
Epoch 1/20
2657/2657 [==============================] - 54s 17ms/step - loss: 0.4667 - accuracy: 0.8320 - val_loss: 0.1253 - val_accuracy: 0.9594
Epoch 2/20
2657/2657 [==============================] - 45s 16ms/step - loss: 0.3241 - accuracy: 0.8803 - val_loss: 0.0252 - val_accuracy: 0.9936
Epoch 3/20
2657/2657 [==============================] - 46s 16ms/step - loss: 0.3025 - accuracy: 0.8894 - val_loss: 0.0398 - val_accuracy: 0.9866
Epoch 4/20
2657/2657 [==============================] - 47s 17ms/step - loss: 0.2955 - accuracy: 0.8936 - val_loss: 0.0533 - val_accuracy: 0.9818
Epoch 5/20
2657/2657 [==============================] - 45s 16ms/step - loss: 0.3018 - accuracy: 0.8924 - val_loss: 0.0064 - val_accuracy: 0.9984
Epoch 6/20
2657/2657 [===================

In [4]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ---- Paths & data ----
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/Cervical Cancer"
IMG_SIZE = (256, 256)   
BATCH_SIZE = 8
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED, shuffle=True
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED, shuffle=False
)

# Split val/test 50/50 from the validation subset
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance (same as second snippet) =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation (same as second snippet) =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Normalization (same as second snippet) =====
# NOTE: We use simple Rescaling(1./255) for ALL models, including transfer learning.
# This matches the second snippet; it's fine for your experiments.
def normalize_block(x):
    return layers.Rescaling(1./255)(x)

# =========================================
# Classifier head (shared for TL models)
# =========================================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# =========================================
# 1) Baseline CNN (from scratch)
#   - keep architecture flexible, but preprocessing matches second snippet
# =========================================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# =========================================
# 2) ResNet50 (prebuilt)
# =========================================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# =========================================
# 3) VGG16 (prebuilt)
# =========================================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# =========================================
# 4) InceptionV3 (prebuilt)
# =========================================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# =========================================
# 5) EfficientNetB0 (prebuilt)
# =========================================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# =========================================
# Pick a model to train (same interface)
# =========================================
MODEL_NAME = "efficientnet"   # one of: "cnn", "resnet", "vgg", "inception", "efficientnet"
WEIGHTS = "imagenet"          # "imagenet" or None
FINE_TUNE_BASE = False        # set True to unfreeze backbone later

input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Found 25000 files belonging to 5 classes.
Using 21250 files for training.
Found 25000 files belonging to 5 classes.
Using 3750 files for validation.
Classes: ['cervix_dyk', 'cervix_koc', 'cervix_mep', 'cervix_pab', 'cervix_sfi']
Epoch 1/20
2657/2657 [==============================] - 50s 15ms/step - loss: 1.6508 - accuracy: 0.1984 - val_loss: 1.7657 - val_accuracy: 0.0000e+00
Epoch 2/20
2657/2657 [==============================] - 39s 14ms/step - loss: 1.6492 - accuracy: 0.2051 - val_loss: 1.9595 - val_accuracy: 0.0000e+00
Epoch 3/20
2657/2657 [==============================] - 41s 15ms/step - loss: 1.6470 - accuracy: 0.2016 - val_loss: 1.4176 - val_accuracy: 1.0000
Epoch 4/20
2657/2657 [==============================] - 44s 15ms/step - loss: 1.6466 - accuracy: 0.2049 - val_loss: 1.3073 - val_accuracy: 1.0000
Epoch 5/20


2025-09-24 23:20:58.804100: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 316 of 1000


   8/2657 [..............................] - ETA: 42s - loss: 1.5933 - accuracy: 0.2344    

2025-09-24 23:21:07.478015: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] Shuffle buffer filled.


2657/2657 [==============================] - 59s 15ms/step - loss: 1.6470 - accuracy: 0.2030 - val_loss: 1.1289 - val_accuracy: 1.0000
Epoch 6/20
2657/2657 [==============================] - 43s 15ms/step - loss: 1.6487 - accuracy: 0.2004 - val_loss: 1.0325 - val_accuracy: 1.0000
Epoch 7/20
2657/2657 [==============================] - 41s 15ms/step - loss: 1.6492 - accuracy: 0.2019 - val_loss: 1.7396 - val_accuracy: 0.0000e+00
efficientnetb0  |  Test acc: 1.0000  |  Test loss: 1.4176
